# Huấn luyện Auxiliary Density Head cho YOLO-World
Notebook này được thiết kế để chạy trên Kaggle nhằm huấn luyện nhánh **Density Head** (dùng cho thuật toán Density-Guided Soft-NMS) trong khi đóng băng (freeze) trọng số của YOLO-World.

**Lưu ý:** Đảm bảo bạn đã Mount bộ dữ liệu `FSC-147` vào đường dẫn `/kaggle/input/fsc147`.

In [ ]:
!pip install -q ultralytics opencv-python

## 1. Import Thư Viện & Cấu Hình Type Hints

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import numpy as np
import cv2
from typing import List, Tuple, Dict, Optional, Any
from ultralytics import YOLOWorld

# ==========================================
# CONFIGURATION & KAGGLE PATHS
# ==========================================
DATA_DIR: str = "/kaggle/input/fsc147"  # Cập nhật theo tên dataset trên Kaggle của bạn
IMAGE_DIR: str = os.path.join(DATA_DIR, "images_384_VarV2")
SPLIT_PATH: str = os.path.join(DATA_DIR, "Train_Test_Val_FSC_147.json")
ANNO_PATH: str = os.path.join(DATA_DIR, "annotation_FSC147_384.json")

BATCH_SIZE: int = 8
EPOCHS: int = 50
LEARNING_RATE: float = 1e-4
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE: int = 640
FEATURE_STRIDE: int = 16  # YOLO P4 stride

print(f"Sử dụng thiết bị: {DEVICE}")

## 2. Dataset & Target Generation

In [ ]:
def generate_density_map(image_shape: Tuple[int, int], points: List[List[float]], sigma: float = 4.0) -> np.ndarray:
    """
    Sinh ra Heatmap Mật độ (Density Map) từ tập hợp các điểm Ground Truth.
    
    Args:
        image_shape (Tuple[int, int]): Kích thước heatmap (H, W)
        points (List[List[float]]): Danh sách tọa độ [x, y] của vật thể
        sigma (float): Độ lan tỏa của Gaussian blur
        
    Returns:
        np.ndarray: Density map kích thước (H, W)
    """
    H, W = image_shape
    density_map: np.ndarray = np.zeros((H, W), dtype=np.float32)
    
    if not points:
        return density_map

    for point in points:
        x, y = min(int(point[0]), W - 1), min(int(point[1]), H - 1)
        density_map[y, x] = 1.0
        
    # Áp dụng Gaussian blur để tạo hiệu ứng mật độ lan tỏa
    density_map = cv2.GaussianBlur(density_map, (15, 15), sigma)
    
    # Cân bằng lại sao cho tổng sum heatmap xấp xỉ số lượng vật thể
    total_sum = density_map.sum()
    if total_sum > 0:
        density_map = (density_map / total_sum) * len(points)
        
    return density_map

class FSC147Dataset(Dataset):
    def __init__(self, split: str = "train") -> None:
        super().__init__()
        if not os.path.exists(SPLIT_PATH):
            raise FileNotFoundError(f"Không tìm thấy {SPLIT_PATH}. Hãy kiểm tra lại DATA_DIR.")
            
        with open(SPLIT_PATH, 'r') as f:
            self.images: List[str] = json.load(f).get(split, [])
            
        with open(ANNO_PATH, 'r') as f:
            self.annos: Dict[str, Any] = json.load(f)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_name = self.images[idx]
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        image = Image.open(img_path).convert("RGB")
        W, H = image.size
        
        # Resize ảnh về chuẩn YOLO 640x640
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
        img_tensor: torch.Tensor = TF.to_tensor(image)  # Shape: (3, 640, 640)
        
        # Chuyển đổi tọa độ Ground Truth Points theo tỷ lệ Resize
        gt_points: List[List[float]] = self.annos.get(img_name, {}).get("points", [])
        scaled_points: List[List[float]] = [
            [p[0] * IMAGE_SIZE / W, p[1] * IMAGE_SIZE / H] for p in gt_points
        ]
            
        # Tạo Target Density Map kích thước thu nhỏ (640 / 16 = 40)
        feat_size = IMAGE_SIZE // FEATURE_STRIDE
        feat_points = [[p[0] / FEATURE_STRIDE, p[1] / FEATURE_STRIDE] for p in scaled_points]
        
        density_map: np.ndarray = generate_density_map((feat_size, feat_size), feat_points)
        density_tensor: torch.Tensor = torch.from_numpy(density_map).unsqueeze(0)  # Shape: (1, 40, 40)
        
        return img_tensor, density_tensor

## 3. Kiến Trúc Mô Hình (Hybrid Architecture)

In [ ]:
class DensityHead(nn.Module):
    """
    Mạng CNN Auxiliary nội suy Heatmap Mật Độ từ Feature Map.
    """
    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1)  # Trả về 1 kênh (Density Map)
        )
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        density_map = self.conv_layers(x)
        return torch.nn.functional.relu(density_map)  # Đảm bảo density >= 0


class DensityYOLOWorld(nn.Module):
    """
    Lớp bọc ngoài (Wrapper) YOLO-World, thực hiện Forward Hook để lấy Features.
    """
    def __init__(self, yolo_weights: str = "yolov8s-world.pt") -> None:
        super().__init__()
        self.yolo = YOLOWorld(yolo_weights)
        
        # Đóng băng Backbone & Detection Head của YOLO-World
        for param in self.yolo.parameters():
            param.requires_grad = False
            
        self.features: Optional[torch.Tensor] = None
        self._register_hook()
        
        # P4 feature layer thường có 256 channels
        self.density_head = DensityHead(in_channels=256)

    def _register_hook(self) -> None:
        def hook_fn(module: nn.Module, input_args: Tuple[torch.Tensor, ...]) -> None:
            # YOLO layer nhận tuple input. input_args[0] thường là List[Tensors] từ FPN
            features_list = input_args[0]
            if isinstance(features_list, (list, tuple)) and len(features_list) >= 3:
                # P3 (stride 8), P4 (stride 16), P5 (stride 32)
                self.features = features_list[1]  # Chọn P4
            else:
                self.features = features_list

        # Gắn Hook vào lớp Cuối (Detect Layer) của YOLO
        detect_layer = self.yolo.model.model[-1]
        detect_layer.register_forward_pre_hook(hook_fn)

    def forward(self, x: torch.Tensor) -> Tuple[Any, torch.Tensor]:
        # Backbone Forward -> Kích hoạt Hook
        raw_preds = self.yolo.model(x)
        
        if self.features is None:
            raise RuntimeError("Hook không bắt được Feature Map. Kiểm tra lại cấu trúc Model YOLOv8.")
            
        density_map = self.density_head(self.features)
        return raw_preds, density_map

    def train_mode(self) -> None:
        self.yolo.eval()  # Backbone luôn Freeze (chế độ Eval)
        self.density_head.train()
        
    def eval_mode(self) -> None:
        self.yolo.eval()
        self.density_head.eval()

## 4. Vòng Lặp Huấn Luyện (Training Loop)

In [ ]:
def train() -> None:
    print("Khởi tạo mô hình Hybrid Density YOLO-World...")
    model = DensityYOLOWorld("yolov8s-world.pt").to(DEVICE)
    model.train_mode()
    
    print("Chuẩn bị dữ liệu...")
    try:
        train_dataset = FSC147Dataset(split="train")
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    except Exception as e:
        print(f"LỖI tải dữ liệu: {e}")
        return
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.density_head.parameters(), lr=LEARNING_RATE)
    
    print(f"Bắt đầu Training với {len(train_loader)} batches mỗi epoch...")
    
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        model.density_head.train()
        
        for batch_idx, (images, gt_density) in enumerate(train_loader):
            images = images.to(DEVICE)
            gt_density = gt_density.to(DEVICE)
            
            optimizer.zero_grad()
            _, pred_density = model(images)
            
            loss = criterion(pred_density, gt_density)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"  Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.6f}")
                
        avg_loss = epoch_loss / len(train_loader)
        print(f"=> HOÀN THÀNH EPOCH {epoch+1} | Average Loss: {avg_loss:.6f}")

    print("Huấn luyện thành công! Đang lưu mô hình...")
    torch.save(model.density_head.state_dict(), "density_head_best.pth")
    print("Đã lưu trọng số vào file: density_head_best.pth")

In [ ]:
# Chạy huấn luyện (Bỏ comment dòng dưới để chạy thực tế trên Kaggle)
# train()